# Day 17 · 七種分工方式：Multi-Agent Workflow Patterns

> 第三部・戰術編排　|　✅ 完整可執行

**前置需求**：🔑 需要 Gemini API 金鑰

**對應文章**：`Day 17 - 七種分工方式：Multi-Agent Workflow Patterns.md`

## 今天要學會

1. 認得七種多 agent 模式並說出各自的適用場合
2. 把每個模式對應到具體的 ADK 元件
3. ⚠️ 知道**「誰決定下一步」**才是這七種模式真正的分類軸

## 環境設定

每一天都是獨立的，這段設定刻意重複，讓你可以從任何一天開始。

In [1]:
import warnings

warnings.filterwarnings("ignore")

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared import ask, get_model, new_session, peek_state, print_state, quiet, run_once

quiet()

import google.adk

print("google-adk", google.adk.__version__)

google-adk 2.8.0


## 1. 七種模式，其實只在問一件事

網路上的「多 agent 模式」文章通常給你七八個名字配七八張圖，
背起來很累，因為它沒告訴你分類軸是什麼。

分類軸只有一條：**下一步由誰決定？**

| # | 模式 | 下一步由誰決定 | ADK 元件 |
|---|---|---|---|
| 1 | Prompt Chaining | **你**（寫死的順序） | `SequentialAgent` / 圖的直線邊 |
| 2 | Routing | **模型**或**你的程式** | `sub_agents` 交棒 / `ctx.route` |
| 3 | Parallelization | **你**（一次全發） | `ParallelAgent` / fan-out + `JoinNode` |
| 4 | Orchestrator–Workers | **模型**（挑工具） | `AgentTool` |
| 5 | Evaluator–Optimizer | **模型**（喊停） | `LoopAgent` + `exit_loop` |
| 6 | Hierarchical Delegation | **模型**（逐層往下） | 巢狀 `sub_agents` |
| 7 | Human-in-the-Loop | **人** | `require_confirmation` |

看懂這一欄，七個模式就只剩三類：**你決定（1、3）**、
**模型決定（2、4、5、6）**、**人決定（7）**。

模型決定的那四個彈性最大，但也是最容易失控、最難測試的——
這正是 Day 13 的 Graph 存在的理由。

In [2]:
# 七個模式用到的元件，全部先確認 import 得到
from google.adk import Workflow
from google.adk.agents import LlmAgent, LoopAgent, ParallelAgent, SequentialAgent
from google.adk.runners import InMemoryRunner
from google.adk.tools import FunctionTool, exit_loop
from google.adk.tools.agent_tool import AgentTool
from google.adk.workflow import START, node

COMPONENTS = [
    ("1 Prompt Chaining", SequentialAgent),
    ("2 Routing", Workflow),
    ("3 Parallelization", ParallelAgent),
    ("4 Orchestrator-Workers", AgentTool),
    ("5 Evaluator-Optimizer", LoopAgent),
    ("6 Hierarchical", LlmAgent),
    ("7 Human-in-the-Loop", FunctionTool),
]
for label, cls in COMPONENTS:
    print(f"  ✅ {label:26s} → {cls.__name__}")

  ✅ 1 Prompt Chaining          → SequentialAgent
  ✅ 2 Routing                  → Workflow
  ✅ 3 Parallelization          → ParallelAgent
  ✅ 4 Orchestrator-Workers     → AgentTool
  ✅ 5 Evaluator-Optimizer      → LoopAgent
  ✅ 6 Hierarchical             → LlmAgent
  ✅ 7 Human-in-the-Loop        → FunctionTool


## 2. 模式 1：Prompt Chaining

**下一步由你決定。** 步驟寫死，前一站的輸出餵給後一站。

適合：流程本來就固定的加工線（翻譯 → 潤稿、抽取 → 格式化）。
不適合：需要看情況轉彎的任務。

In [3]:
translator = LlmAgent(
    name="translator",
    model=get_model(),
    instruction="把使用者的中文翻成英文，只回譯文。",
    output_key="en",
)
shortener = LlmAgent(
    name="shortener",
    model=get_model(),
    instruction="把這句英文改寫成推文長度（15 字以內），只回結果：\n\n{en?}",
    output_key="tweet",
)

chain = SequentialAgent(name="chain", sub_agents=[translator, shortener])

r = InMemoryRunner(agent=chain, app_name="day17")
sid = await new_session(r)
await ask(r, "我們的新功能可以讓報表產出時間從三小時縮短到五分鐘。", session_id=sid)
print_state(await peek_state(r, sid))

  en: Our new feature reduces report generation time from three hours to five minutes.
  tweet: Reports: 3 hours ➔ 5 mins!


## 3. 模式 2：Routing

**下一步由「模型」或「你的程式」決定**——這是本日最需要分清楚的一格。

ADK 給你兩條路，差別在**誰做判斷**。

### 2-a. 讓模型判斷：`sub_agents` 交棒

把 agent 掛成 `sub_agents`，ADK 會自動加上 `transfer_to_agent` 工具。
**模型讀 `description` 決定要轉給誰**——所以 `description` 不是註解，是路由表。

In [4]:
billing = LlmAgent(
    name="billing",
    model=get_model(),
    description="處理帳單、發票、退費、付款方式的問題",
    instruction="你是帳務專員，用一句話回覆，繁體中文。",
)
tech = LlmAgent(
    name="tech",
    model=get_model(),
    description="處理登入失敗、當機、功能異常等技術問題",
    instruction="你是技術支援，用一句話回覆，繁體中文。",
)

front_desk = LlmAgent(
    name="front_desk",
    model=get_model(),
    instruction="你是客服總機。判斷問題類型後轉給對應的專員，不要自己回答。",
    sub_agents=[billing, tech],
)

r2 = InMemoryRunner(agent=front_desk, app_name="day17")
for q in ["我上個月被重複扣款兩次", "App 一直閃退打不開"]:
    sid2 = await new_session(r2)
    print(f"📥 {q}")
    await ask(r2, q, session_id=sid2, trace=True)
    print()

📥 我上個月被重複扣款兩次


  🔧 [front_desk] 呼叫 transfer_to_agent({'agent_name': 'billing'})
  ↩️  [front_desk] transfer_to_agent 回傳 {'result': None}


  💬 [billing] 了解您的狀況後，我會立即為您查詢並協助處理上個月重複扣款的退費事宜。

📥 App 一直閃退打不開


  🔧 [front_desk] 呼叫 transfer_to_agent({'agent_name': 'tech'})
  ↩️  [front_desk] transfer_to_agent 回傳 {'result': None}


  💬 [tech] 請嘗試將 App 更新至最新版本或重新安裝，若問題持續請提供您的手機型號與作業系統版本以便我們進一步協助。



### 2-b. 讓程式判斷：圖的 `ctx.route`

同樣是路由，但判斷寫在 Python 裡。**沒有模型呼叫，結果 100% 可重現。**

> ⚠️ **這裡有個新手一定會踩的坑：使用者輸入的那句話「不會」自動變成節點參數。**
> `@node` 預設 `parameter_binding="state"`，參數是拿名字去 **session state** 裡找的（Day 14）。
> 所以要嘛前面接一個有 `output_key` 的 `LlmAgent`，要嘛像下面這樣**建 session 時就把值放進 state**。
> 直接寫 `def route(ctx, raw_text)` 會得到
> `Missing value for parameter ... not found in state`。

In [5]:
from google.adk.agents.context import Context
from google.genai import types


@node
def route(ctx: Context, q: str) -> str:
    """q 來自 state["q"]——純字串比對，沒有模型參與。"""
    ctx.route = "billing" if any(k in q for k in ("扣款", "發票", "退費")) else "tech"
    return f"判定為 {ctx.route}"


@node
def to_billing() -> str:
    return "→ 帳務組受理"


@node
def to_tech() -> str:
    return "→ 技術組受理"


router_graph = Workflow(
    name="router_graph",
    edges=[
        (START, route),
        (route, {"billing": to_billing, "tech": to_tech}),
    ],
)


async def run_graph(wf, text):
    runner = InMemoryRunner(agent=wf, app_name="day17")
    sid = await new_session(runner, state={"q": text})     # ← 把輸入放進 state
    msg = types.Content(role="user", parts=[types.Part(text=text)])
    async for ev in runner.run_async(user_id="student", session_id=sid, new_message=msg):
        out = getattr(ev, "output", None)
        if out is not None:
            print("  ▪", out)


for q in ["我上個月被重複扣款兩次", "App 一直閃退打不開"]:
    print(f"📥 {q}")
    await run_graph(router_graph, q)

📥 我上個月被重複扣款兩次
  ▪ 判定為 billing
  ▪ → 帳務組受理
📥 App 一直閃退打不開
  ▪ 判定為 tech
  ▪ → 技術組受理


**兩種路由怎麼選**

| | 2-a 模型交棒 | 2-b 圖的 `ctx.route` |
|---|---|---|
| 判斷者 | 模型讀 `description` | 你的 Python 程式 |
| 成本 | 多一次模型呼叫 | 零 |
| 可重現 | ❌ 同樣輸入可能轉去不同人 | ✅ 100% |
| 能處理沒想到的輸入 | ✅ | ❌ 只認你寫的規則 |
| 可單元測試 | 很難 | 容易 |

**規則列得完就用 2-b。** 列不完、或使用者講話很自由，才交給模型。

## 4. 模式 3：Parallelization

**下一步由你決定**，只是一次發全部。分支之間**看不到彼此的 state**（Day 16）。

In [6]:
def reviewers():
    """工廠函式——agent 只能有一個父節點，不能重用實例。"""
    return [
        LlmAgent(name="pros", model=get_model(), output_key="pros",
                 instruction="列出這個提案的兩個優點，條列式，繁體中文。"),
        LlmAgent(name="cons", model=get_model(), output_key="cons",
                 instruction="列出這個提案的兩個風險，條列式，繁體中文。"),
    ]


fan_out_join = SequentialAgent(
    name="fan_out_join",
    sub_agents=[
        ParallelAgent(name="both", sub_agents=reviewers()),      # fan-out
        LlmAgent(name="judge", model=get_model(), output_key="verdict",
                 instruction="根據優點 {pros?} 與風險 {cons?} 給一句結論，繁體中文。"),
    ],                                                            # join
)

r3 = InMemoryRunner(agent=fan_out_join, app_name="day17")
sid3 = await new_session(r3)
await ask(r3, "提案：全公司改用四天工作制。", session_id=sid3)
print("結論:", (await peek_state(r3, sid3)).get("verdict"))

結論: 實施四天工作制雖能有效提升員工滿意度並吸引人才，但公司必須審慎評估產能下降與客戶服務斷層的潛在風險，透過完善的配套措施在工作效率與生活平衡之間取得最佳平衡。


## 5. 模式 4：Orchestrator–Workers

**下一步由模型決定**，但決定的方式是「挑工具」而不是「交棒」。

用 `AgentTool` 把 sub-agent 包成工具。跟模式 2-a 的關鍵差別：
**控制權會回到總指揮手上**——工具回傳後主 agent 繼續講話。

In [7]:
zh_writer = LlmAgent(name="zh_writer", model=get_model(),
                     description="寫繁體中文文案", instruction="寫一句繁體中文廣告標語，只回標語。")
en_writer = LlmAgent(name="en_writer", model=get_model(),
                     description="寫英文文案", instruction="Write one English ad slogan. Reply with the slogan only.")

orchestrator = LlmAgent(
    name="orchestrator",
    model=get_model(),
    instruction=(
        "你是行銷總監。使用者要雙語文案時，"
        "**兩個工具都要呼叫**，最後把兩句並排整理給使用者。"
    ),
    tools=[AgentTool(agent=zh_writer), AgentTool(agent=en_writer)],
)

print(await run_once(orchestrator, "幫「會自己記帳的 App」想中英各一句標語。", trace=True))

  🔧 [orchestrator] 呼叫 zh_writer({'request': '幫「會自己記帳的 App」想一句中文標語，強調自動化、省時、無痛記帳'})
  🔧 [orchestrator] 呼叫 en_writer({'request': 'Write an English slogan for an automated expense tracker app that logs expenses automatically, saves time, and makes budgeting effortless.'})


  ↩️  [orchestrator] zh_writer 回傳 {'result': '動動手指太累？讓錢包自己說真心話。'}
  ↩️  [orchestrator] en_writer 回傳 {'result': 'Spend less time tracking, more living.'}


  💬 [orchestrator] 這裡為您準備的中英雙語標語：

* **中文標語：** 動動手指太累？讓錢包自己說真心話。
* **英文標語：** Spend less time tracking, more living.
這裡為您準備的中英雙語標語：

* **中文標語：** 動動手指太累？讓錢包自己說真心話。
* **英文標語：** Spend less time tracking, more living.


## 6. 模式 5：Evaluator–Optimizer

**下一步由模型決定**，決定的是「還要不要再一輪」。
⚠️ `LoopAgent` 自己不會停——停止是 sub_agent 呼叫 `exit_loop` 的責任（Day 16）。

In [8]:
writer = LlmAgent(
    name="writer", model=get_model(), output_key="title",
    instruction=("為一篇講「工程師如何管理技術債」的文章下標題。\n"
                 "有評語就照著改，沒有就寫第一版。只回標題。\n\n"
                 "前一版：{title?}\n評語：{note?}"),
)
evaluator = LlmAgent(
    name="evaluator", model=get_model(), output_key="note", tools=[exit_loop],
    instruction=("評審這個標題：{title?}\n\n"
                 "同時滿足「20 字以內」「不含『秘訣』『終極』這類農場字眼」時，"
                 "**呼叫 exit_loop 工具**；否則用一句話說最該改什麼。"),
)

loop = LoopAgent(name="refine", sub_agents=[writer, evaluator], max_iterations=3)
r5 = InMemoryRunner(agent=loop, app_name="day17")
sid5 = await new_session(r5)
await ask(r5, "開始", session_id=sid5, trace=True)
print("\n最終標題:", (await peek_state(r5, sid5)).get("title"))

  💬 [writer] 重構還是放推？工程師的技術債生存指南


  🔧 [evaluator] 呼叫 exit_loop({})
  ↩️  [evaluator] exit_loop 回傳 {'result': None}

最終標題: 重構還是放推？工程師的技術債生存指南


## 7. 模式 6：Hierarchical Delegation

交棒可以**疊很多層**。總機轉給組長，組長再轉給組員。

⚠️ 每多一層就多一次模型呼叫，而且**每一層都可能轉錯**。
三層以下還能除錯，再深就該考慮換成圖。

In [9]:
refund = LlmAgent(name="refund", model=get_model(),
                  description="專門處理退費金額試算",
                  instruction="你是退費專員，用一句話說明退費金額怎麼算，繁體中文。")
invoice = LlmAgent(name="invoice", model=get_model(),
                   description="專門處理發票開立與更正",
                   instruction="你是發票專員，用一句話回覆，繁體中文。")

billing_lead = LlmAgent(
    name="billing_lead", model=get_model(),
    description="帳務組組長，底下有退費與發票兩位專員",
    instruction="你是帳務組長。判斷該給退費還是發票專員，轉過去，不要自己回答。",
    sub_agents=[refund, invoice],
)

top = LlmAgent(
    name="top", model=get_model(),
    instruction="你是總機。帳務相關一律轉給 billing_lead，不要自己回答。",
    sub_agents=[billing_lead],
)

r6 = InMemoryRunner(agent=top, app_name="day17")
sid6 = await new_session(r6)
await ask(r6, "我要退訂，想知道能退多少錢", session_id=sid6, trace=True)

  🔧 [top] 呼叫 transfer_to_agent({'agent_name': 'billing_lead'})
  ↩️  [top] transfer_to_agent 回傳 {'result': None}


  🔧 [billing_lead] 呼叫 transfer_to_agent({'agent_name': 'refund'})
  ↩️  [billing_lead] transfer_to_agent 回傳 {'result': None}


  💬 [refund] 退費金額會依據您目前的方案類型、已使用天數比例以及相關退費規定來進行試算。


'退費金額會依據您目前的方案類型、已使用天數比例以及相關退費規定來進行試算。'

## 8. 模式 7：Human-in-the-Loop

**下一步由人決定。** 工具標上 `require_confirmation=True`，
執行會停在那裡等人核准（完整流程見 Day 15）。

In [10]:
EXECUTED = []


def issue_refund(user_id: str, amount: int) -> dict:
    """實際執行退款。

    Args:
        user_id: 使用者代號。
        amount: 退款金額。
    """
    EXECUTED.append((user_id, amount))
    return {"refunded": amount}


hitl = LlmAgent(
    name="hitl", model=get_model(),
    instruction="使用者要退款就呼叫 issue_refund。",
    tools=[FunctionTool(func=issue_refund, require_confirmation=True)],
)

r7 = InMemoryRunner(agent=hitl, app_name="day17")
sid7 = await new_session(r7)
msg = types.Content(role="user", parts=[types.Part(text="幫 u-501 退款 800 元")])
async for ev in r7.run_async(user_id="student", session_id=sid7, new_message=msg):
    for p in (ev.content.parts if ev.content else []):
        if p.function_call:
            print("  🔧 function_call:", p.function_call.name)

print("\n工具執行了嗎？", EXECUTED or "沒有——卡在等人核准")

  🔧 function_call: issue_refund
  🔧 function_call: adk_request_confirmation

工具執行了嗎？ 沒有——卡在等人核准


## 9. 📌 文章沒講到的補充：Python 沒有 `RoutedAgent`

文章的 Agent Routing 補充框寫的是 `RoutedAgent` / `RoutedLlm`。
**那是 TypeScript 專屬的 API，Python SDK 裡不存在。**

照抄會直接 `ImportError`：

In [11]:
import pkgutil

import google.adk

hits = [m.name for m in pkgutil.walk_packages(google.adk.__path__, "google.adk.") if "Routed" in m.name]
print("ADK Python 裡叫 Routed* 的模組:", hits or "一個都沒有")

try:
    from google.adk.agents import RoutedAgent  # noqa: F401
    print("import 成功")
except ImportError as exc:
    print(f"❌ {exc}")

print("\n→ Python 的等價做法就是本日的 2-a（模型交棒）與 2-b（ctx.route）。")
print("   已驗證存在的路由開關：",
      [f for f in LlmAgent.model_fields if "transfer" in f])

ADK Python 裡叫 Routed* 的模組: 一個都沒有
❌ cannot import name 'RoutedAgent' from 'google.adk.agents' (/Users/linshihuan/Dev/github/adk_tutor/.venv/lib/python3.13/site-packages/google/adk/agents/__init__.py)

→ Python 的等價做法就是本日的 2-a（模型交棒）與 2-b（ctx.route）。
   已驗證存在的路由開關： ['disallow_transfer_to_parent', 'disallow_transfer_to_peers']


### 附帶一提：交棒是可以鎖的

`disallow_transfer_to_parent` / `disallow_transfer_to_peers` 這兩個欄位
決定專員能不能把球再踢回去。鎖死之後，使用者換個話題就會卡在專員手上。

In [12]:
locked = LlmAgent(
    name="locked_billing", model=get_model(),
    description="只處理帳單",
    instruction="你是帳務專員，只回答帳務問題，繁體中文一句話。",
    disallow_transfer_to_parent=True,
    disallow_transfer_to_peers=True,
)
print("locked_billing 還能把球踢回去嗎？")
print("  → parent:", not locked.disallow_transfer_to_parent)
print("  → peers :", not locked.disallow_transfer_to_peers)
print("\n⚠️ 鎖死的後果：使用者接著問技術問題，會被這位專員硬答，而不是轉給 tech。")

locked_billing 還能把球踢回去嗎？
  → parent: False
  → peers : False

⚠️ 鎖死的後果：使用者接著問技術問題，會被這位專員硬答，而不是轉給 tech。


## 10. 常見錯誤與踩坑

| 症狀 | 原因 |
|---|---|
| 總機自己回答，不轉給任何人 | `instruction` 沒明講「不要自己回答」 |
| 一直轉錯人 | **`description` 才是路由表**，寫太籠統模型就猜錯 |
| `Agent 'x' already has a parent agent` | 一個 agent 只能有一個父節點，改用工廠函式 |
| 交棒之後就回不來了 | 這是交棒的**設計**；要回得來請改用 `AgentTool`（模式 4） |
| 照文章寫 `RoutedAgent` 就 `ImportError` | 那是 TypeScript API，Python 沒有 |
| 圖的路由節點抱怨 `not found in state` | 使用者輸入**不會**自動變參數，要先進 state（Day 14） |
| 階層太深，出錯不知道錯在哪一層 | 每層都是一次模型判斷，超過三層改用圖 |
| `LoopAgent` 停不下來 | 沒人呼叫 `exit_loop`，只剩 `max_iterations` 擋著 |

## 11. 動手練習

1. 把 2-a 的 `billing` 和 `tech` 的 `description` 都改成「處理客戶問題」，
   重跑兩個問題，看模型多常轉錯——體會 `description` 就是路由表。
2. 把 2-b 的 `route` 節點改成用 `LlmAgent` 判斷類別，
   比較「同一個問題連跑三次」的結果穩不穩定。
3. 把模式 4 的 `AgentTool` 改寫成 `sub_agents` 交棒，
   確認總指揮再也沒辦法把中英兩句「並排整理」給你。
4. 在模式 6 的階層中間再插一層，數數看一個問題總共觸發幾次模型呼叫。

## 本日回顧

- 七種模式只有一條分類軸：**下一步由誰決定**——你、模型、還是人。
- **`description` 是路由表，不是註解。** 交棒轉錯人，九成是它寫得太籠統。
- **交棒（`sub_agents`）控制權不回來；`AgentTool` 會回來。** 這是模式 2 與模式 4 的分水嶺。
- ⚠️ **Python 沒有 `RoutedAgent` / `RoutedLlm`**，那是 TypeScript 專屬；
  等價做法是模型交棒或圖的 `ctx.route`。
- **規則列得完就別讓模型路由**——`ctx.route` 零成本、可重現、可測試。
- 模型決定的模式彈性最大，也最難測試；這正是 Day 13 圖的存在理由。

---
**下一天 → `../day18_collaborative_workflows/`**